In [ ]:
# train_dir = '/Users/umjirana/tomatoleaf/tomato/train'
# val_dir = '/Users/umjirana/tomatoleaf/tomato/val'

In [ ]:
import os
from pathlib import Path

import numpy as np
import tensorflow as tf
from IPython.display import display
from PIL import Image
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image


In [ ]:
# 훈련용 이미지 데이터셋을 로드하고, 정규화하는 코드

train_data = tf.keras.utils.image_dataset_from_directory(
    '/Users/umjirana/tomatoleaf/tomato/train',  # 학습용 이미지가 저장된 디렉토리 경로
    labels='inferred',                        # 디렉토리 이름을 라벨로 자동 추출 (폴더명이 클래스 이름)
    label_mode='categorical',                 # 라벨을 one-hot 인코딩된 형태로 반환
    image_size=(256, 256),                    # 이미지를 256x256 크기로 리사이즈
    batch_size=32)                            # 한 번에 가져올 배치 크기 설정 (32장씩 묶어서 처리)

train_data = train_data.map(lambda x, y: (x / 255.0, y))  # 이미지 픽셀 값을 0~1 범위로 정규화 (정규화된 이미지와 라벨 반환)

In [ ]:
# 검증용 이미지 데이터셋을 로드하고, 정규화하는 코드

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    '/Users/umjirana/tomatoleaf/tomato/val',  # 검증용 이미지가 저장된 디렉토리 경로
    labels='inferred',                      # 디렉토리 이름을 기반으로 라벨을 자동 추출 (폴더명이 클래스 이름)
    label_mode='categorical',               # 라벨을 one-hot 인코딩된 형식으로 반환
    image_size=(256, 256),                  # 모든 이미지를 256x256 크기로 리사이즈
    batch_size=32)                          # 배치 단위로 32장씩 묶어서 처리

val_data = val_data.map(lambda x, y: (x / 255.0, y))  # 이미지 픽셀 값을 0~255에서 0~1 범위로 정규화

In [ ]:
# 탐색적 데이터 분석 EDA 과정
path = Path('/Users/umjirana/tomatoleaf/tomato/train/Tomato___Target_Spot')
image_files = [p for p in path.iterdir() if p.is_file()][:6]

for image_path in image_files:
    print(image_path.name)
    display(Image.open(image_path).resize((256, 256)))


In [ ]:
# 사전 학습된 DenseNet121 모델을 불러와 특징 추출기로 사용
conv_base = DenseNet121(
    weights='imagenet',          # ImageNet 데이터셋으로 사전 학습된 가중치 사용
    include_top=False,           # 최상단 분류기 층은 제외 (사용자 정의 출력층을 추가할 예정)
    input_shape=(256, 256, 3),   # 입력 이미지 크기 (256x256 RGB 이미지)
    pooling='avg'                # GlobalAveragePooling2D 적용하여 출력 벡터로 변환
)

In [ ]:
conv_base.trainable = False  # conv_base의 가중치를 학습하지 않도록 고정

In [ ]:
model = Sequential()  # 순차적으로 레이어를 쌓는 Sequential 모델 생성
model.add(conv_base)  # 사전 학습된 DenseNet121 모델 추가 (이미지 특징 추출기 역할)
model.add(BatchNormalization())  # 배치 정규화: 학습 안정성 및 속도 향상
model.add(Dense(256, activation='relu'))  # 은닉층 (노드 256개, ReLU 활성화 함수)
model.add(Dropout(0.35))  # 과적합 방지를 위한 드롭아웃 (35% 확률로 뉴런 비활성화)
model.add(BatchNormalization())  # 다시 배치 정규화
model.add(Dense(120, activation='relu'))  # 두 번째 은닉층 (노드 120개, ReLU 함수 사용)
model.add(Dense(10, activation='softmax'))  # 출력층 (클래스 10개 분류, 소프트맥스로 확률 출력)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),             # Adam 최적화 알고리즘 사용 (학습률 0.0001로 설정)
    loss='categorical_crossentropy',       # 다중 클래스 분류를 위한 손실 함수 (라벨이 one-hot 인코딩일 때 사용)
    metrics=['accuracy']                   # 모델 평가 지표로 정확도(accuracy) 사용
)

In [ ]:
# 모델 학습
history = model.fit(
    train_data,                          # 학습에 사용할 데이터셋 (train_data)
    epochs=5,                          # 최대 100 에폭(epoch) 동안 학습
    validation_data=val_data,           # 학습 중 성능 평가를 위한 검증 데이터셋
    callbacks=[EarlyStopping(patience=0)]  # 조기 종료 설정: 성능이 더 이상 개선되지 않으면 바로 학습 중단
)

In [ ]:
# 검증 데이터로 모델 평가
evaluation = model.evaluate(val_data)

# 평가 지표 출력
print("검증 손실(Validation Loss):", evaluation[0])
print("검증 정확도(Validation Accuracy):", evaluation[1])

In [ ]:
# 추가 학습 (6~10번째 에폭 수행)
history_additional = model.fit(
    train_data,
    epochs=10,                # 총 에폭 목표: 10
    initial_epoch=5,          # 이전 학습이 0~4였으므로 5부터 시작
    validation_data=val_data,
    callbacks=[EarlyStopping(patience=0)]
)

# 추가 학습 후 다시 평가
evaluation = model.evaluate(val_data)

# 평가 지표 출력
print("검증 손실(Validation Loss):", evaluation[0])
print("검증 정확도(Validation Accuracy):", evaluation[1])

In [ ]:
# 대시보드에서 사용할 모델 저장
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / 'densenet121_tomato.keras'
model.save(MODEL_PATH)
print(f'모델 저장 완료: {MODEL_PATH}')


In [ ]:
class_names = [
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

In [ ]:
# 예측할 이미지 경로
img_path = '/Users/umjirana/Desktop/python/tomato23.jpg'  # 예시

img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
predicted_class = int(np.argmax(prediction))
confidence = float(np.max(prediction))

print(f'예측: {class_names[predicted_class]} ({confidence * 100:.2f}%)')
display(img)


In [ ]:
# 예측할 이미지 경로
img_path = '/Users/umjirana/Desktop/python/Tomato_Yellow.jpg'  # 예시

img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
predicted_class = int(np.argmax(prediction))
confidence = float(np.max(prediction))

print(f'예측: {class_names[predicted_class]} ({confidence * 100:.2f}%)')
display(img)
